In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from scipy.stats import norm

from utils import delta_call, svi_iv, nn_iv, heston_iv

In [2]:
df = pd.read_csv("clean_iv_surface_input.csv")
df['date'] = pd.to_datetime(df['date'])

r = 0.04
q = 0

df['w'] = (df['IV']**2)*df['T']
df["F"] = df["underlying_price"] * np.exp((r - q) * df["T"])
df["k"] = np.log(df["strike"] / df["F"])


start = '2026-05-11'
end = "2026-06-12"


# get stock values of AAPL

import yfinance as yf

stock_prices = yf.download(
    "SPY",
    start=start,
    end = pd.Timestamp(end)+ pd.Timedelta(days=1),
    auto_adjust=True
)

stock_prices = stock_prices['Close'].copy()

dates = sorted(df["date"].unique())
dates.append(pd.Timestamp(end))

stock_path = [stock_prices.loc[date] for date in dates]
stock_path = np.asarray(stock_path, dtype=float)

# options that we can hedge 

call_option_df = df[
    (df["date"] == start) & 
    (df['expiry'] == end) &
    (df["option_type"] == "call")
    ]

[*********************100%***********************]  1 of 1 completed


In [17]:
spot_price = call_option_df["underlying_price"].unique()

In [16]:
call_option_df['strike'].unique()

array([675., 676., 680., 685., 687., 689., 690., 692., 695., 697., 700.,
       705., 708., 709., 710., 712., 713., 715., 716., 717., 719., 720.,
       721., 722., 723., 724., 725., 726., 727., 728., 729., 730., 731.,
       732., 733., 734., 735., 736., 737., 738., 739., 740., 741., 742.,
       743., 744., 745., 746., 747., 748., 749., 750., 755., 760., 765.,
       770., 775., 780., 785., 790., 795., 800., 805., 810.])

In [42]:
call_option_df = df[
    (df["date"] == start) & 
    (df['expiry'] == end) &
    (df["option_type"] == "call")
    ]


call_option_df = call_option_df[
    (call_option_df['strike'] < float(spot_price + 15)) &
    (call_option_df['strike'] > float(spot_price - 15))
].copy()

len(call_option_df)

/var/folders/5k/t6kz0v_x2gqbmd_6mhqwtbjh0000gn/T/ipykernel_2185/81295627.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  (call_option_df['strike'] < float(spot_price + 15)) &
/var/folders/5k/t6kz0v_x2gqbmd_6mhqwtbjh0000gn/T/ipykernel_2185/81295627.py:10: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  (call_option_df['strike'] > float(spot_price - 15))


28

In [43]:
results = []

models = ["bs","svi", "nn"]   # add more later

for model in models:

    for opt_idx in range(len(call_option_df)):

        option = call_option_df.iloc[opt_idx]

        S = stock_path.ravel()

        sigma_sticky = option["IV"]
        
        K = option["strike"]
        T = option["T"]
        k = option["k"]
        r = 0.04
        C0 = option["market_price"]

        n_steps = len(S) - 1
        dt = T / n_steps
        times = np.linspace(0, T, n_steps + 1)


        if model == "bs":
            sigmas_call = sigma_sticky * np.ones(n_steps)


        if model == "svi":
            sigmas_call = np.array([
                svi_iv(dates[i], K, S[i], T - times[i])
                for i in range(n_steps)
            ], dtype=float)

        if model == 'heston':
            sigmas_call = np.array([
                heston_iv(dates[i], S[i], K, T - times[i], r)
                for i in range(n_steps)
            ], dtype=float)


        if model == "nn":
            sigmas_call = np.array([
                nn_iv(str(dates[i]), k, T - times[i], S[i], is_call = 1)
                for i in range(n_steps)
            ], dtype = float)
        

        deltas_call = delta_call(
            S[:n_steps],
            K,
            (T - times)[:n_steps],
            sigmas_call
        )

        call_payout_discounted = np.maximum(S[-1] - K, 0) * np.exp(-r * T)

        stock_profit = np.sum(
            (S[1:] - S[:-1] * np.exp(r * dt))
            * np.exp(-r * times[1:])
            * deltas_call
        )

        hedging_error = call_payout_discounted - stock_profit - C0
        relative_error = abs(hedging_error) / C0

        results.append({
            "model": model,
            "option_index": opt_idx,
            "strike": K,
            "T": T,
            "market_price": C0,
            "hedging_error": hedging_error,
            "relative_error": relative_error,
        })

results_df = pd.DataFrame(results)

In [44]:
summary_df = (
    results_df
    .groupby("model")
    .agg(
        mean_abs_error=("hedging_error", lambda x: np.mean(np.abs(x))),
        rmse=("hedging_error", lambda x: np.sqrt(np.mean(x**2))),
        mean_relative_error=("relative_error", "mean"),
        std_relative_error=("relative_error", "std"),
        median_relative_error=("relative_error", "median"),
        max_relative_error=("relative_error", "max"),
    )
    .reset_index()
)

summary_df

,model,mean_abs_error,rmse,mean_relative_error,std_relative_error,median_relative_error,max_relative_error
0,bs,4.900784,5.111904,0.296788,0.086848,0.284249,0.457147
1,nn,4.593429,4.903477,0.271416,0.101091,0.253936,0.447129
2,svi,5.844510,6.071355,0.357805,0.105192,0.357703,0.546915
